In [2]:

!nvidia-smi
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


Sun May 10 21:21:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             47W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
# ==============================================================================
# CELL 1: Setup Environment and Mount Google Drive
# ==============================================================================
!pip install -q ultralytics

from google.colab import drive
import os
from ultralytics import YOLO

# Mount Google Drive to save our weights permanently
drive.mount('/content/drive')

# Define permanent paths in your Drive
PROJECT_DIR = '/content/drive/MyDrive/Smart_Scan/Detection_Model'
DATASET_DIR = os.path.join(PROJECT_DIR, 'dataset')  # Singular: dataset folder
RUNS_DIR = os.path.join(PROJECT_DIR, 'yolo_runs')

os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

print(f"✅ Environment ready. Saving weights to: {RUNS_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Environment ready. Saving weights to: /content/drive/MyDrive/Smart_Scan/Detection_Model/yolo_runs


In [4]:
import os, shutil, yaml, glob
from pathlib import Path

# ==============================================================================
# CELL 2: IBEM -> YOLO Conversion (with Drive cache for fast re-runs)
# On first run: converts and saves to Drive.
# On re-runs:   copies from Drive cache directly (skips conversion).
# ==============================================================================

EXTRACT_PATH = os.path.join(DATASET_DIR, 'IBEM_data')
LOCAL_EXTRACT = '/content/IBEM_data'
LOCAL_YOLO    = '/content/IBEM_yolo_v2'
DRIVE_YOLO    = os.path.join(DATASET_DIR, 'IBEM_yolo_v2')  # permanent cache

DATA_YAML = os.path.join(LOCAL_YOLO, 'data.yaml')

# ---- FAST RE-RUN PATH: load from Drive cache --------------------------------
if os.path.exists(DRIVE_YOLO) and os.path.exists(os.path.join(DRIVE_YOLO, 'data.yaml')):
    if not os.path.exists(LOCAL_YOLO):
        print('[CACHE HIT] Copying converted YOLO dataset from Drive -> /content/ ...')
        shutil.copytree(DRIVE_YOLO, LOCAL_YOLO)
        print('[OK] Done. Skipped full IBEM conversion.')
    else:
        print('[CACHE HIT] YOLO dataset already in /content/ -- ready instantly.')

    # Fix paths in data.yaml to point to local content
    with open(DATA_YAML, 'r') as f:
        cfg = yaml.safe_load(f)
    cfg['path'] = LOCAL_YOLO
    with open(DATA_YAML, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False)

    print(f'[OK] DATA_YAML = {DATA_YAML}')
    print('     Run Cell 3 to train!')

# ---- FIRST RUN PATH: full conversion + save to Drive cache ------------------
else:
    print('[FIRST RUN] No Drive cache found. Running full IBEM -> YOLO conversion...')

    # Step 1: Copy IBEM_data from Drive -> /content/ if needed
    if not os.path.exists(LOCAL_EXTRACT) or len(os.listdir(LOCAL_EXTRACT)) == 0:
        print('[INFO] Copying IBEM_data from Drive -> /content/ ...')
        shutil.copytree(EXTRACT_PATH, LOCAL_EXTRACT)
        print('[OK] Copy done.')
    else:
        print('[OK] /content/IBEM_data already exists locally.')

    # Step 2: Group Tr*/Va*/Ts* folders
    split_map = {}
    for folder in sorted(os.listdir(LOCAL_EXTRACT)):
        fpath = os.path.join(LOCAL_EXTRACT, folder)
        if not os.path.isdir(fpath):
            continue
        if folder.startswith('Tr'):
            split_map.setdefault('train', []).append(fpath)
        elif folder.startswith('Va'):
            split_map.setdefault('val', []).append(fpath)
        elif folder.startswith('Ts'):
            split_map.setdefault('test', []).append(fpath)

    for split, folders in split_map.items():
        print(f'   {split}: {[os.path.basename(f) for f in folders]}')

    # Step 3: Create YOLO dirs (clean slate)
    if os.path.exists(LOCAL_YOLO):
        shutil.rmtree(LOCAL_YOLO)
    for split in ['train', 'val', 'test']:
        os.makedirs(os.path.join(LOCAL_YOLO, 'images', split), exist_ok=True)
        os.makedirs(os.path.join(LOCAL_YOLO, 'labels', split), exist_ok=True)

    # Step 4: Converter (IBEM % format -> YOLO normalized 0-1)
    def ibem_to_yolo(txt_path):
        yolo_lines = []
        with open(txt_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                parts = line.split()
                if len(parts) < 4:
                    continue
                try:
                    x_rel = float(parts[0])
                    y_rel = float(parts[1])
                    w_pct = float(parts[2])
                    h_pct = float(parts[3])
                    cls   = int(float(parts[4])) if len(parts) >= 5 else 0
                    x_c = max(0.0, min(1.0, (x_rel + w_pct / 2) / 100.0))
                    y_c = max(0.0, min(1.0, (y_rel + h_pct / 2) / 100.0))
                    w   = max(0.001, min(1.0, w_pct / 100.0))
                    h   = max(0.001, min(1.0, h_pct / 100.0))
                    yolo_lines.append(f'{cls} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}')
                except (ValueError, IndexError):
                    continue
        return yolo_lines

    # Step 5: Convert all splits
    counters    = {'train': 0, 'val': 0, 'test': 0}
    total_boxes = {'train': 0, 'val': 0, 'test': 0}
    no_annot    = 0

    print('[INFO] Converting with FIXED format...')
    for split, folders in split_map.items():
        for folder in folders:
            for jpg_path in glob.glob(os.path.join(folder, '*.jpg')):
                base = os.path.splitext(os.path.basename(jpg_path))[0]
                txt_path = os.path.join(folder, base + '.txt')
                if not os.path.exists(txt_path):
                    alt = os.path.join(folder, base.replace('-page', '-color_page') + '.txt')
                    txt_path = alt if os.path.exists(alt) else None
                dst_img = os.path.join(LOCAL_YOLO, 'images', split, os.path.basename(jpg_path))
                shutil.copy2(jpg_path, dst_img)
                dst_lbl = os.path.join(LOCAL_YOLO, 'labels', split, base + '.txt')
                if txt_path:
                    lines = ibem_to_yolo(txt_path)
                    with open(dst_lbl, 'w') as f:
                        f.write('\n'.join(lines))
                    total_boxes[split] += len(lines)
                else:
                    open(dst_lbl, 'w').close()
                    no_annot += 1
                counters[split] += 1

    print('[OK] Conversion results:')
    for split in ['train', 'val', 'test']:
        n = counters[split]
        b = total_boxes[split]
        print(f'   {split:6s}: {n} images | {b} boxes | {b/n:.1f} boxes/img avg' if n > 0 else f'   {split}: 0 images')
    print(f'   Images with no annotation: {no_annot}')

    # Step 6: Create data.yaml
    data_yaml_content = {
        'path':  LOCAL_YOLO,
        'train': 'images/train',
        'val':   'images/val',
        'test':  'images/test',
        'nc':    2,
        'names': ['embedded', 'isolated']
    }
    with open(DATA_YAML, 'w') as f:
        yaml.dump(data_yaml_content, f, default_flow_style=False)

    # Step 7: Save converted dataset to Drive for future re-runs
    print('[INFO] Saving converted YOLO dataset to Drive cache (one-time)...')
    if os.path.exists(DRIVE_YOLO):
        shutil.rmtree(DRIVE_YOLO)
    shutil.copytree(LOCAL_YOLO, DRIVE_YOLO)
    print(f'[OK] Saved to Drive: {DRIVE_YOLO}')
    print(f'[OK] DATA_YAML = {DATA_YAML}')
    print('     Next re-run will load from Drive cache instantly!')
    print('     Run Cell 3 to train!')


✅ /content/IBEM_data already exists.
   train: ['Tr00', 'Tr01', 'Tr10']
   test: ['Ts00', 'Ts01', 'Ts10', 'Ts11']
   val: ['Va00', 'Va01']

⚙️  Converting with FIXED format...

✅ Conversion results:
   train : 5171 images | 103939 boxes | 20.1 boxes/image avg
   val   : 957 images | 18370 boxes | 19.2 boxes/image avg
   test  : 2144 images | 44383 boxes | 20.7 boxes/image avg
   Images with no paired annotation: 0

✅ data.yaml → /content/IBEM_yolo_v2/data.yaml
   nc=2, classes: embedded (inline), isolated (display)

📍 DATA_YAML = /content/IBEM_yolo_v2/data.yaml
▶️  Run Cell 3 to retrain with CORRECT annotations!


In [5]:
# ==============================================================================
# CELL 3: Auto-Resume Training Loop (OPTIMIZED FOR A100)
# ==============================================================================
import os
from ultralytics import YOLO

LOCAL_YOLO = '/content/IBEM_yolo_v2'
DATA_YAML  = os.path.join(LOCAL_YOLO, 'data.yaml')

if not os.path.exists(DATA_YAML):
    raise SystemExit("❌ DATA_YAML not found. Run Cell 2 first!")

print(f"✅ DATA_YAML found: {DATA_YAML}")

# Use a new run name to avoid conflicts
RUN_NAME = 'math_detector'
LAST_CHECKPOINT = os.path.join(RUNS_DIR, RUN_NAME, 'weights', 'last.pt')

if os.path.exists(LAST_CHECKPOINT):
    print(f"\n🔄 Resuming from checkpoint: {LAST_CHECKPOINT}")
    model = YOLO(LAST_CHECKPOINT)
    results = model.train(resume=True)
    print("✅ Training resumed and completed!")
else:
    print("\n🚀 Starting fresh training...")

    # UPGRADE 1: Use Small model instead of Nano (much better accuracy)
    model = YOLO('yolov8s.pt')

    results = model.train(
        data=DATA_YAML,
        epochs=100,             # Increased epochs for better learning
        imgsz=640,

        # --- A100 UTILIZATION UPGRADES ---
        batch=128,              # FORCED 128: AutoBatch played it too safe. This will use ~35GB VRAM.
        cache=True,             # Caches images in system RAM (you have 84GB, this makes it SUPER fast)
        workers=16,             # Uses more CPU cores to load data faster
        # ---------------------------------

        project=RUNS_DIR,
        name=RUN_NAME,
        save=True,
        save_period=5,
        device=0,
        exist_ok=True
    )
    print("🎉 Training complete!")


✅ DATA_YAML found: /content/IBEM_yolo_v2/data.yaml

🚀 Starting fresh training...
Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/IBEM_yolo_v2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=math_dete

In [6]:
import os

# Updated to look for the correct folder
MODEL_DIR = os.path.join(RUNS_DIR, 'math_detector')
weights_dir = os.path.join(MODEL_DIR, 'weights')

print("🔍 Verifying training results...\n")

if os.path.exists(weights_dir):
    weights_files = os.listdir(weights_dir)
    print(f"✅ Weights folder: {weights_dir}")

    has_best = 'best.pt' in weights_files
    has_last = 'last.pt' in weights_files
    print(f"\n   best.pt : {'✅' if has_best else '❌'}")
    print(f"   last.pt : {'✅' if has_last else '❌'}")

    if has_best:
        best_path = os.path.join(weights_dir, 'best.pt')
        size_mb = os.path.getsize(best_path) / 1e6
        print(f"\n🎉 best.pt ready! Size: {size_mb:.1f} MB")
        print(f"\n📥 DOWNLOAD COMMAND:")
        print(f"   from google.colab import files")
        print(f"   files.download('{best_path}')")
else:
    print(f"❌ Weights folder not found at: {weights_dir}")


🔍 Verifying training results...

✅ Weights folder: /content/drive/MyDrive/Smart_Scan/Detection_Model/yolo_runs/math_detector/weights

   best.pt : ✅
   last.pt : ✅

🎉 best.pt ready! Size: 22.5 MB

📥 DOWNLOAD COMMAND:
   from google.colab import files
   files.download('/content/drive/MyDrive/Smart_Scan/Detection_Model/yolo_runs/math_detector/weights/best.pt')
